<a href="https://colab.research.google.com/github/AbhinavKumar07/datathon_2026/blob/main/datathon2026_nb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

#Problem: How can we help new (small buisness) restaurant owners open up their business in areas that are optimized for traffic

#Get datasets --> df
df_consumer_data = pd.read_csv('./ConsumerData.csv')
df_migration_data = pd.read_csv('./OC2025_ZIPMigration.csv')
df_us_address_Data = pd.read_csv('./USAddressData.csv')
df_zip_data = pd.read_csv('./ZipData.csv')
df_property_val_data = pd.read_csv('./property_value.csv')


#df_consumer_data.head()
#df_migration_data.head()
#df_us_address_Data.head()
#df_zip_data.head()
#df_property_val_data.sample(10)

### Handling Missing Values in `df_consumer_data`

Based on the `df_consumer_data.info()` output, many columns have missing values. We will fill these with appropriate values. For 'Y'/'N' type columns, `NaN` will be replaced with 'N'. For numerical columns, we'll use the median.

In [ ]:
# Fill NaN values for 'Y'/'N' type columns with 'N'
consumer_data_columns = [
    'Charitable', 'Health', 'Political', 'Religious', 'Veteran', 'SingleParent',
    'GrandChildren', 'CatOwner', 'DogOwner', 'CreditCardUser', 'SelfImprovement',
    'MusicCollector', 'MovieCollector', 'Photography', 'AutoWork', 'Fishing',
    'CampingHiking', 'HuntingShooting', 'Gardening', 'EnvironmentalIssues',
    'HomeImprovement', 'HomeImprovementDIY', 'OutdoorsGrouping',
    'InvestmentsForeign', 'BeautyCosmetics', 'TVCable', 'WirelessCellularPhoneOwner',
    'EducationOnline'
]
for col in consumer_data_columns:
    df_consumer_data[col] = df_consumer_data[col].fillna('N')

# Fill NaN values for MaritalStatus with 'Unknown'
df_consumer_data['MaritalStatus'] = df_consumer_data['MaritalStatus'].fillna('Unknown')

# Fill NaN values for OwnerRenter with 'Unknown' or the mode
# Let's check value counts first to decide for OwnerRenter
# print(df_consumer_data['OwnerRenter'].value_counts())
# Assuming 'O' (Owner) is the mode or a reasonable default for missing values, or 'Unknown'
df_consumer_data['OwnerRenter'] = df_consumer_data['OwnerRenter'].fillna('Unknown')


# Fill NaN values for numerical columns with their median
# HomePurchaseDate, NumberOfChildren, HouseholdSize, NetWorth, VehicleKnownOwnedNumber
numeric_columns = [
    'HomePurchaseDate', 'NumberOfChildren', 'HouseholdSize', 'NetWorth',
    'VehicleKnownOwnedNumber'
]
for col in numeric_columns:
    df_consumer_data[col] = df_consumer_data[col].fillna(df_consumer_data[col].median())


print(df_consumer_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 44 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   RecordID                    10000 non-null  int64  
 1   MAK                         10000 non-null  int64  
 2   BaseMak                     10000 non-null  int64  
 3   Address                     10000 non-null  object 
 4   City                        10000 non-null  object 
 5   State                       10000 non-null  object 
 6   Zipcode                     10000 non-null  int64  
 7   Latitude                    10000 non-null  float64
 8   Longitude                   10000 non-null  float64
 9   OwnerRenter                 10000 non-null  object 
 10  HomePurchaseDate            10000 non-null  float64
 11  Charitable                  10000 non-null  object 
 12  Health                      10000 non-null  object 
 13  Political                   1000

In [ ]:
#Column index numbers

column_indices = {}
for col_name in consumer_data_columns:
    if col_name in df_consumer_data.columns:
        column_indices[col_name] = df_consumer_data.columns.get_loc(col_name)
    else:
        column_indices[col_name] = 'Not Found'

for col, index in column_indices.items():
    print(f"Column '{col}': Index {index}")

Column 'Charitable': Index 11
Column 'Health': Index 12
Column 'Political': Index 13
Column 'Religious': Index 14
Column 'Veteran': Index 15
Column 'SingleParent': Index 17
Column 'GrandChildren': Index 19
Column 'CatOwner': Index 21
Column 'DogOwner': Index 22
Column 'CreditCardUser': Index 24
Column 'SelfImprovement': Index 26
Column 'MusicCollector': Index 27
Column 'MovieCollector': Index 28
Column 'Photography': Index 29
Column 'AutoWork': Index 30
Column 'Fishing': Index 31
Column 'CampingHiking': Index 32
Column 'HuntingShooting': Index 33
Column 'Gardening': Index 34
Column 'EnvironmentalIssues': Index 35
Column 'HomeImprovement': Index 36
Column 'HomeImprovementDIY': Index 37
Column 'OutdoorsGrouping': Index 38
Column 'InvestmentsForeign': Index 39
Column 'BeautyCosmetics': Index 40
Column 'TVCable': Index 41
Column 'WirelessCellularPhoneOwner': Index 42
Column 'EducationOnline': Index 43


### Cleaning `df_consumer_data`

First, let's inspect the `df_consumer_data` to understand its structure, data types, and identify potential issues like missing values or irrelevant columns.

In [ ]:
print(df_consumer_data.info())
print(df_consumer_data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 44 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   RecordID                    10000 non-null  int64  
 1   MAK                         10000 non-null  int64  
 2   BaseMak                     10000 non-null  int64  
 3   Address                     10000 non-null  object 
 4   City                        10000 non-null  object 
 5   State                       10000 non-null  object 
 6   Zipcode                     10000 non-null  int64  
 7   Latitude                    10000 non-null  float64
 8   Longitude                   10000 non-null  float64
 9   OwnerRenter                 10000 non-null  object 
 10  HomePurchaseDate            10000 non-null  float64
 11  Charitable                  10000 non-null  object 
 12  Health                      10000 non-null  object 
 13  Political                   1000

### Cleaning `df_migration_data` from the start

Let's begin by inspecting the `df_migration_data` to understand its structure, data types, and identify any initial cleaning needs.

In [ ]:
print(df_migration_data.info())
print(df_migration_data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ZipCode            85 non-null     int64  
 1   State              85 non-null     object 
 2   FIPS               85 non-null     int64  
 3   MovesOutOfZip      85 non-null     int64  
 4   MovesIntoZip       85 non-null     int64  
 5   TotalMoves         85 non-null     int64  
 6   PctLeave           85 non-null     float64
 7   PctMoveIn          85 non-null     float64
 8   ZipCode_Latitude   85 non-null     float64
 9   ZipCode_Longitude  85 non-null     float64
dtypes: float64(4), int64(5), object(1)
memory usage: 6.8+ KB
None
   ZipCode State  FIPS  MovesOutOfZip  MovesIntoZip  TotalMoves  PctLeave  \
0    92655    CA  6059             75            76         151     49.67   
1    92844    CA  6059            238           183         421     56.53   
2    90630    CA  6059 

### Detailed Cleaning and Feature Engineering for `df_migration_data`

Since the initial inspection showed no missing values, we'll proceed with checking for duplicates and creating a new feature to enhance its utility for the AI model.

In [ ]:
# Check for duplicate rows
duplicates_migration = df_migration_data.duplicated().sum()
print(f"Number of duplicate rows in df_migration_data: {duplicates_migration}")

# If duplicates exist, you might choose to drop them:
# if duplicates_migration > 0:
#     df_migration_data.drop_duplicates(inplace=True)
#     print(f"Duplicates removed. New number of rows: {len(df_migration_data)}")

# Create a 'NetMigration' column (MovesIntoZip - MovesOutOfZip)
# This indicates population growth or decline in a zip code, relevant for restaurant traffic.
df_migration_data['NetMigration'] = df_migration_data['MovesIntoZip'] - df_migration_data['MovesOutOfZip']

print('\nDataFrame info after adding NetMigration column:')
print(df_migration_data.info())
print('\nFirst 5 rows of df_migration_data with NetMigration:')
print(df_migration_data.head())

Number of duplicate rows in df_migration_data: 0

DataFrame info after adding NetMigration column:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ZipCode            85 non-null     int64  
 1   State              85 non-null     object 
 2   FIPS               85 non-null     int64  
 3   MovesOutOfZip      85 non-null     int64  
 4   MovesIntoZip       85 non-null     int64  
 5   TotalMoves         85 non-null     int64  
 6   PctLeave           85 non-null     float64
 7   PctMoveIn          85 non-null     float64
 8   ZipCode_Latitude   85 non-null     float64
 9   ZipCode_Longitude  85 non-null     float64
 10  NetMigration       85 non-null     int64  
dtypes: float64(4), int64(6), object(1)
memory usage: 7.4+ KB
None

First 5 rows of df_migration_data with NetMigration:
   ZipCode State  FIPS  MovesOutOfZip  MovesIntoZip  

### Data Consistency Checks for `df_migration_data`

Before considering outliers, let's perform some consistency checks on the numerical columns to ensure the data adheres to expected relationships (e.g., `TotalMoves` being the sum of `MovesOutOfZip` and `MovesIntoZip`).

In [ ]:
# Check consistency of TotalMoves
df_migration_data['CalculatedTotalMoves'] = df_migration_data['MovesOutOfZip'] + df_migration_data['MovesIntoZip']
mismatched_total_moves = df_migration_data[df_migration_data['TotalMoves'] != df_migration_data['CalculatedTotalMoves']]

if not mismatched_total_moves.empty:
    print("Inconsistencies found in 'TotalMoves' column:")
    display(mismatched_total_moves[['MovesOutOfZip', 'MovesIntoZip', 'TotalMoves', 'CalculatedTotalMoves']])
else:
    print("No inconsistencies found in 'TotalMoves' column.")

# Check consistency of PctLeave and PctMoveIn (should sum up to ~100)
df_migration_data['CalculatedPctTotal'] = df_migration_data['PctLeave'] + df_migration_data['PctMoveIn']
mismatched_percentages = df_migration_data[abs(df_migration_data['CalculatedPctTotal'] - 100) > 0.01] # Allowing for small floating point discrepancies

if not mismatched_percentages.empty:
    print("\nInconsistencies found in 'PctLeave' and 'PctMoveIn' percentages:")
    display(mismatched_percentages[['PctLeave', 'PctMoveIn', 'CalculatedPctTotal']])
else:
    print("\nNo significant inconsistencies found in 'PctLeave' and 'PctMoveIn' percentages.")

# Drop the temporary calculated columns
df_migration_data.drop(columns=['CalculatedTotalMoves', 'CalculatedPctTotal'], inplace=True, errors='ignore')

No inconsistencies found in 'TotalMoves' column.

No significant inconsistencies found in 'PctLeave' and 'PctMoveIn' percentages.


### Cleaning `df_us_address_Data`

Let's start by inspecting the `df_us_address_Data` to understand its structure, data types, and identify any initial cleaning needs.

### Cleaning `df_zip_data`

Let's start by inspecting the `df_zip_data` to understand its structure, data types, and identify any initial cleaning needs.

In [ ]:
print(df_zip_data.info())
print(df_zip_data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 34 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   RecordID                              704 non-null    int64  
 1   ZipCode                               704 non-null    int64  
 2   City                                  704 non-null    object 
 3   County                                704 non-null    object 
 4   State                                 704 non-null    object 
 5   Latitude                              704 non-null    float64
 6   Longitude                             704 non-null    float64
 7   DominantAreaCode                      704 non-null    int64  
 8   ResidentialDeliveries                 704 non-null    int64  
 9   ResidentialPOBoxes                    704 non-null    int64  
 10  BusinessDeliveries                    704 non-null    int64  
 11  BusinessPOBoxes    

### Detailed Cleaning and Missing Value Imputation for `df_us_address_Data`

Based on the inspection, `StreetPreDirection` and `StreetPostDirection` are entirely null, so we will drop them. For other columns with missing values, we will use 'Unknown' for object types and the median for numerical types, similar to our approach with `df_consumer_data`.

In [ ]:
# Drop columns with all NaN values
df_us_address_Data.drop(columns=['StreetPreDirection', 'StreetPostDirection'], inplace=True)

# Fill NaN values for object/categorical columns with 'Unknown'
object_columns_to_fill = ['BaseMAK', 'Suite', 'StreetSuffix', 'SuiteType', 'SuiteNumber', 'PlaceCode']
for col in object_columns_to_fill:
    # Only fill if the column exists and is of object type
    if col in df_us_address_Data.columns and df_us_address_Data[col].dtype == 'object':
        df_us_address_Data[col] = df_us_address_Data[col].fillna('Unknown')
    # Handle BaseMAK which is float but often acts as an ID, converting to object then filling
    elif col == 'BaseMAK' and df_us_address_Data[col].dtype == 'float64':
        df_us_address_Data['BaseMAK'] = df_us_address_Data['BaseMAK'].fillna(0).astype(int).astype(str) # Fill with 0, convert to int, then to string for 'Unknown' consistency
    elif col == 'PlaceCode' and df_us_address_Data[col].dtype == 'float64':
        df_us_address_Data['PlaceCode'] = df_us_address_Data['PlaceCode'].fillna(0).astype(int).astype(str)

# Fill NaN values for numerical columns with their median
# ElevationInMeters is the primary numerical column with NaNs
if 'ElevationInMeters' in df_us_address_Data.columns:
    df_us_address_Data['ElevationInMeters'] = df_us_address_Data['ElevationInMeters'].fillna(df_us_address_Data['ElevationInMeters'].median())

print(df_us_address_Data.info())
print(df_us_address_Data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18999 entries, 0 to 18998
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RecordID            18999 non-null  int64  
 1   MAK                 18999 non-null  int64  
 2   BaseMAK             18999 non-null  object 
 3   Address             18999 non-null  object 
 4   Suite               18999 non-null  object 
 5   City                18999 non-null  object 
 6   State               18999 non-null  object 
 7   HouseNumber         18999 non-null  object 
 8   StreetName          18999 non-null  object 
 9   StreetSuffix        18999 non-null  object 
 10  SuiteType           18999 non-null  object 
 11  SuiteNumber         18999 non-null  object 
 12  ZipCode             18999 non-null  int64  
 13  ZipCodePlus4        18999 non-null  object 
 14  Latitude            18999 non-null  float64
 15  Longitude           18999 non-null  float64
 16  Cens

In [ ]:
print(df_us_address_Data.info())
print(df_us_address_Data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18999 entries, 0 to 18998
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RecordID            18999 non-null  int64  
 1   MAK                 18999 non-null  int64  
 2   BaseMAK             18999 non-null  object 
 3   Address             18999 non-null  object 
 4   Suite               18999 non-null  object 
 5   City                18999 non-null  object 
 6   State               18999 non-null  object 
 7   HouseNumber         18999 non-null  object 
 8   StreetName          18999 non-null  object 
 9   StreetSuffix        18999 non-null  object 
 10  SuiteType           18999 non-null  object 
 11  SuiteNumber         18999 non-null  object 
 12  ZipCode             18999 non-null  int64  
 13  ZipCodePlus4        18999 non-null  object 
 14  Latitude            18999 non-null  float64
 15  Longitude           18999 non-null  float64
 16  Cens

In [ ]:
# Fill NaN values for 'LastLineIndicator' with 'Unknown'
df_zip_data['LastLineIndicator'] = df_zip_data['LastLineIndicator'].fillna('Unknown')

# Fill NaN values for numerical columns with their median
numeric_zip_columns = [
    'MedianHouseholdIncome', 'PerCapitaIncome', 'MedianHomeValue', 'MedianAge',
    'MedianAgeMale', 'MedianAgeFemale'
]
for col in numeric_zip_columns:
    df_zip_data[col] = df_zip_data[col].fillna(df_zip_data[col].median())

print(df_zip_data.info())
print(df_zip_data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 34 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   RecordID                              704 non-null    int64  
 1   ZipCode                               704 non-null    int64  
 2   City                                  704 non-null    object 
 3   County                                704 non-null    object 
 4   State                                 704 non-null    object 
 5   Latitude                              704 non-null    float64
 6   Longitude                             704 non-null    float64
 7   DominantAreaCode                      704 non-null    int64  
 8   ResidentialDeliveries                 704 non-null    int64  
 9   ResidentialPOBoxes                    704 non-null    int64  
 10  BusinessDeliveries                    704 non-null    int64  
 11  BusinessPOBoxes    

### Data Integration: Preparing Datasets for Merging

In [ ]:
# Step 1: Standardize 'Zipcode' column name in df_consumer_data to 'ZipCode'
df_consumer_data.rename(columns={'Zipcode': 'ZipCode'}, inplace=True)

# Step 2: Aggregate df_consumer_data by ZipCode
# We'll calculate various summary statistics for consumer demographics at the zip code level.
# This will prevent an overly large dataset when merging with other zip-level data.

agg_consumer_data = df_consumer_data.groupby('ZipCode').agg(
    ConsumerCount=('RecordID', 'count'),
    AvgHouseholdSize=('HouseholdSize', 'mean'),
    AvgNumberOfChildren=('NumberOfChildren', 'mean'),
    AvgNetWorth=('NetWorth', 'mean'),
    AvgVehicleKnownOwned=('VehicleKnownOwnedNumber', 'mean'),
    # Count 'Y' for binary categorical features
    PctCharitable=('Charitable', lambda x: (x == 'Y').mean() * 100),
    PctHealth=('Health', lambda x: (x == 'Y').mean() * 100),
    PctPolitical=('Political', lambda x: (x == 'Y').mean() * 100),
    PctReligious=('Religious', lambda x: (x == 'Y').mean() * 100),
    PctVeteran=('Veteran', lambda x: (x == 'Y').mean() * 100),
    PctSingleParent=('SingleParent', lambda x: (x == 'Y').mean() * 100),
    PctGrandChildren=('GrandChildren', lambda x: (x == 'Y').mean() * 100),
    PctCatOwner=('CatOwner', lambda x: (x == 'Y').mean() * 100),
    PctDogOwner=('DogOwner', lambda x: (x == 'Y').mean() * 100),
    PctCreditCardUser=('CreditCardUser', lambda x: (x == 'Y').mean() * 100)
).reset_index()

print("Aggregated df_consumer_data info:")
print(agg_consumer_data.info())
print("\nAggregated df_consumer_data head:")
print(agg_consumer_data.head())

Aggregated df_consumer_data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ZipCode               1 non-null      int64  
 1   ConsumerCount         1 non-null      int64  
 2   AvgHouseholdSize      1 non-null      float64
 3   AvgNumberOfChildren   1 non-null      float64
 4   AvgNetWorth           1 non-null      float64
 5   AvgVehicleKnownOwned  1 non-null      float64
 6   PctCharitable         1 non-null      float64
 7   PctHealth             1 non-null      float64
 8   PctPolitical          1 non-null      float64
 9   PctReligious          1 non-null      float64
 10  PctVeteran            1 non-null      float64
 11  PctSingleParent       1 non-null      float64
 12  PctGrandChildren      1 non-null      float64
 13  PctCatOwner           1 non-null      float64
 14  PctDogOwner           1 non-null      float6

### Merging All Datasets by Zip Code

In [ ]:
# Start with df_zip_data as the base, as it contains a broad range of demographic and geographic information.
merged_df = df_zip_data.copy()

# Merge agg_consumer_data
merged_df = pd.merge(merged_df, agg_consumer_data, on='ZipCode', how='left')

# Merge df_migration_data
merged_df = pd.merge(merged_df, df_migration_data, on='ZipCode', how='left', suffixes=('_zip', '_migration'))

# Merge agg_property_data
merged_df = pd.merge(merged_df, agg_property_data, on='ZipCode', how='left')

# Display the info and head of the merged DataFrame
print("Merged DataFrame info:")
print(merged_df.info())
print("\nMerged DataFrame head:")
print(merged_df.head())

Merged DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 63 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   RecordID                              704 non-null    int64  
 1   ZipCode                               704 non-null    int64  
 2   City                                  704 non-null    object 
 3   County                                704 non-null    object 
 4   State_zip                             704 non-null    object 
 5   Latitude                              704 non-null    float64
 6   Longitude                             704 non-null    float64
 7   DominantAreaCode                      704 non-null    int64  
 8   ResidentialDeliveries                 704 non-null    int64  
 9   ResidentialPOBoxes                    704 non-null    int64  
 10  BusinessDeliveries                    704 non-null    int64  
 

### Aggregating `df_property_val_data` by Zip Code

In [ ]:
# Aggregating df_property_val_data by ZipCode
# Calculate average property values for each zip code.

agg_property_data = df_property_val_data.groupby('ZipCode').agg(
    PropertyCount=('RecordId', 'count'),
    AvgFinalValue=('FinalValue', 'mean'),
    AvgHighValue=('HighValue', 'mean'),
    AvgLowValue=('LowValue', 'mean')
).reset_index()

print("Aggregated df_property_val_data info:")
print(agg_property_data.info())
print("\nAggregated df_property_val_data head:")
print(agg_property_data.head())

Aggregated df_property_val_data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ZipCode        92 non-null     int64  
 1   PropertyCount  92 non-null     int64  
 2   AvgFinalValue  92 non-null     float64
 3   AvgHighValue   92 non-null     float64
 4   AvgLowValue    92 non-null     float64
dtypes: float64(3), int64(2)
memory usage: 3.7 KB
None

Aggregated df_property_val_data head:
   ZipCode  PropertyCount  AvgFinalValue  AvgHighValue   AvgLowValue
0    90620          11588   9.499254e+05  1.006192e+06  8.967831e+05
1    90621           5922   1.023487e+06  1.082903e+06  9.673421e+05
2    90623           4053   1.203141e+06  1.272821e+06  1.137261e+06
3    90630          13268   1.110082e+06  1.175201e+06  1.048546e+06
4    90631          14066   9.270143e+05  9.818491e+05  8.752422e+05


### Cleaning `df_property_val_data`

Let's start by inspecting the `df_property_val_data` to understand its structure, data types, and identify any initial cleaning needs, including missing values.

In [ ]:
print(df_property_val_data.info())
print(df_property_val_data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 767204 entries, 0 to 767203
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   RecordId       767204 non-null  int64  
 1   FIPSCode       767204 non-null  int64  
 2   County         767204 non-null  object 
 3   Address        767204 non-null  object 
 4   City           767204 non-null  object 
 5   State          767204 non-null  object 
 6   ZipCode        767204 non-null  int64  
 7   ZipCodePlus4   767204 non-null  int64  
 8   Latitude       767204 non-null  float64
 9   Longitude      767204 non-null  float64
 10  FinalValue     767204 non-null  float64
 11  HighValue      767204 non-null  float64
 12  LowValue       767204 non-null  float64
 13  ValuationDate  767204 non-null  int64  
dtypes: float64(5), int64(5), object(4)
memory usage: 81.9+ MB
None
   RecordId  FIPSCode  County          Address       City State  ZipCode  \
0         1      6059  Or

### Handling Remaining Missing Values in `merged_df`

In [ ]:
# Identify numerical columns with NaNs that should be filled with median
numerical_cols_with_nan = [
    'ConsumerCount', 'AvgHouseholdSize', 'AvgNumberOfChildren', 'AvgNetWorth',
    'AvgVehicleKnownOwned', 'PctCharitable', 'PctHealth', 'PctPolitical',
    'PctReligious', 'PctVeteran', 'PctSingleParent', 'PctGrandChildren',
    'PctCatOwner', 'PctDogOwner', 'PctCreditCardUser',
    'FIPS', 'MovesOutOfZip', 'MovesIntoZip', 'TotalMoves', 'PctLeave', 'PctMoveIn',
    'ZipCode_Latitude', 'ZipCode_Longitude', 'NetMigration',
    'PropertyCount', 'AvgFinalValue', 'AvgHighValue', 'AvgLowValue' # Added property columns
]

for col in numerical_cols_with_nan:
    if col in merged_df.columns:
        # Fill with 0 for counts and moves if absence implies zero activity
        # For averages and percentages, median is more appropriate.
        # Let's use median as a general strategy consistent with previous steps.
        merged_df[col] = merged_df[col].fillna(merged_df[col].median())

# Identify categorical columns with NaNs that should be filled with 'Unknown'
categorical_cols_with_nan = [
    'State_migration' # This was derived from df_migration_data
]

for col in categorical_cols_with_nan:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].fillna('Unknown')

# Drop specified columns as requested by the user
columns_to_drop = ['PctCatOwner', 'PctDogOwner', 'PctHealth']
merged_df.drop(columns=columns_to_drop, errors='ignore', inplace=True)


print("Merged DataFrame info after handling NaNs and dropping columns:")
print(merged_df.info())
print("\nMerged DataFrame head after handling NaNs and dropping columns:")
print(merged_df.head())

Merged DataFrame info after handling NaNs and dropping columns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 60 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   RecordID                              704 non-null    int64  
 1   ZipCode                               704 non-null    int64  
 2   City                                  704 non-null    object 
 3   County                                704 non-null    object 
 4   State_zip                             704 non-null    object 
 5   Latitude                              704 non-null    float64
 6   Longitude                             704 non-null    float64
 7   DominantAreaCode                      704 non-null    int64  
 8   ResidentialDeliveries                 704 non-null    int64  
 9   ResidentialPOBoxes                    704 non-null    int64  
 10  BusinessDeliveries    

In [ ]:
merged_df.head()

,RecordID,ZipCode,City,County,State_zip,Latitude,Longitude,DominantAreaCode,ResidentialDeliveries,ResidentialPOBoxes,...,TotalMoves,PctLeave,PctMoveIn,ZipCode_Latitude,ZipCode_Longitude,NetMigration,PropertyCount,AvgFinalValue,AvgHighValue,AvgLowValue
0,1,90001,FIRESTONE PARK,LOS ANGELES,CA,33.9740,-118.2494,323,13443,304,...,1496.0,55.18,44.82,33.7112,-117.8312,-117.0,9593.0,1.342702e+06,1.420849e+06,1.269169e+06
1,2,90001,LOS ANGELES,LOS ANGELES,CA,33.9740,-118.2494,323,13443,304,...,1496.0,55.18,44.82,33.7112,-117.8312,-117.0,9593.0,1.342702e+06,1.420849e+06,1.269169e+06
2,3,90002,LOS ANGELES,LOS ANGELES,CA,33.9493,-118.2462,323,12443,177,...,1496.0,55.18,44.82,33.7112,-117.8312,-117.0,9593.0,1.342702e+06,1.420849e+06,1.269169e+06
3,4,90002,WATTS,LOS ANGELES,CA,33.9493,-118.2462,323,12443,177,...,1496.0,55.18,44.82,33.7112,-117.8312,-117.0,9593.0,1.342702e+06,1.420849e+06,1.269169e+06
4,5,90003,BROADWAY MANCHESTER,LOS ANGELES,CA,33.9634,-118.2742,323,18149,69,...,1496.0,55.18,44.82,33.7112,-117.8312,-117.0,9593.0,1.342702e+06,1.420849e+06,1.269169e+06


### Final Check for Missing Values in `merged_df`

In [ ]:
print("Number of remaining missing values per column in merged_df:")
print(merged_df.isnull().sum()[merged_df.isnull().sum() > 0])

Number of remaining missing values per column in merged_df:
Series([], dtype: int64)


### Feature Engineering for Restaurant Location Optimization

In [ ]:
# Calculate population density
merged_df['PopulationDensity'] = merged_df['TotalPopulation'] / (merged_df['ResidentialDeliveries'] + merged_df['BusinessDeliveries'])

# Calculate average income per person
merged_df['AvgIncomePerPerson'] = merged_df['PerCapitaIncome']

# Calculate the ratio of business to residential deliveries, which might indicate commercial activity
merged_df['BusinessResidentialRatio'] = merged_df['BusinessDeliveries'] / (merged_df['ResidentialDeliveries'] + 1) # Add 1 to avoid division by zero

# Calculate the most common race
population_columns = [
    'PopulationWhite', 'PopulationAfricanAmerican', 'PopulationAmericanIndianAlaskaNative',
    'PopulationAsian', 'PopulationHispanic', 'PopulationPacificIslander', 'PopulationOther', 'PopulationMultipleRace'
]
# Find the column name with the maximum value for each row among the population columns
merged_df['MostCommonRace'] = merged_df[population_columns].idxmax(axis=1).str.replace('Population', '')

# Add a feature for relative wealth based on median household income and median home value
merged_df['WealthIndex'] = (merged_df['MedianHouseholdIncome'] + merged_df['MedianHomeValue']) / 2

# Education level as a potential indicator of customer base sophistication
merged_df['HigherEducationRatio'] = (merged_df['EducationBachelorsDegree'] + merged_df['EducationAssociatesDegree']) / merged_df['TotalPopulation']

print("Merged DataFrame info after Feature Engineering:")
print(merged_df.info())
print("\nFirst 5 rows of merged_df with new features:")
print(merged_df[['ZipCode', 'PopulationDensity', 'AvgIncomePerPerson', 'BusinessResidentialRatio', 'MostCommonRace', 'WealthIndex', 'HigherEducationRatio']].head())

Merged DataFrame info after Feature Engineering:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 66 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   RecordID                              704 non-null    int64  
 1   ZipCode                               704 non-null    int64  
 2   City                                  704 non-null    object 
 3   County                                704 non-null    object 
 4   State_zip                             704 non-null    object 
 5   Latitude                              704 non-null    float64
 6   Longitude                             704 non-null    float64
 7   DominantAreaCode                      704 non-null    int64  
 8   ResidentialDeliveries                 704 non-null    int64  
 9   ResidentialPOBoxes                    704 non-null    int64  
 10  BusinessDeliveries                   

In [ ]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 66 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   RecordID                              704 non-null    int64  
 1   ZipCode                               704 non-null    int64  
 2   City                                  704 non-null    object 
 3   County                                704 non-null    object 
 4   State_zip                             704 non-null    object 
 5   Latitude                              704 non-null    float64
 6   Longitude                             704 non-null    float64
 7   DominantAreaCode                      704 non-null    int64  
 8   ResidentialDeliveries                 704 non-null    int64  
 9   ResidentialPOBoxes                    704 non-null    int64  
 10  BusinessDeliveries                    704 non-null    int64  
 11  BusinessPOBoxes    

In [ ]:
# Drops the first column (index 0)

#commented cuz it will cause an keyError now
#merged_df = merged_df.drop(columns = ['PctCatOwner' , 'PctDogOwner' , 'PctCharitable' , 'PctHealth' , 'PctReligious' , 'PctVeteran'], axis=1)
#merged_df = merged_df.drop(columns = ['MedianHouseholdIncome' , 'MedianHomeValue'], axis=1)
#merged_df = merged_df.drop(columns = ['EducationBachelorsDegree','EducationAssociatesDegree' , 'BusinessDeliveries' , 'ResidentialDeliveries'], axis=1)
#merged_df = merged_df.drop(columns = ['PopulationWhite' , 'PopulationAfricanAmerican' , 'PopulationAmericanIndianAlaskaNative' , 'PopulationAsian' , 'PopulationHispanic' , 'PopulationPacificIslander' , 'PopulationOther' , 'PopulationMultipleRace'], axis=1)
#merged_df = merged_df.drop(columns = ['EducationNinthGradeOrLess' , 'EducationSomeHighSchool'  , 'EducationSomeCollegeWithoutDiploma'], axis=1)
#merged_df = merged_df.drop(columns = ['ResidentialPOBoxes' , 'BusinessPOBoxes' , 'DominantAreaCode' , 'MedianAgeMale' , 'MedianAgeFemale' , 'EducationHighSchoolGraduate', 'AvgVehicleKnownOwned'], axis=1)
#merged_df = merged_df.drop(columns = ['ZipCode_Latitude' , 'ZipCode_Longitude'], axis=1)
#merged_df = merged_df.drop(columns = ['AvgHighValue' , 'AvgLowValue' , 'PropertyCount' , 'PctVeteran' , 'PctReligious'], axis=1)
#merged_df = merged_df.drop(columns = ['Latitude' , 'Longitude' , 'PerCapitaIncome' , 'AvgNumberOfChildren' , 'AvgNetWorth' , 'PctCharitable'], axis=1)
#merged_df = merged_df.drop(columns = ['PctLeave' , 'PctMoveIn' ,'TotalMoves' , 'PctCreditCardUser' , 'MovesOutOfZip' , 'MovesIntoZip'], axis=1)
#merged_df = merged_df.drop(columns = ['ConsumerCount' , 'PctGrandChildren' , 'State_migration' , 'LastLineIndicator'], axis=1)
#merged_df = merged_df.drop(columns = ['FIPS' , 'PctSingleParent'], axis=1)
#merged_df = merged_df.drop(columns = ['PctPolitical'], axis=1)
#merged_df = merged_df.drop(columns = ['AvgIncomePerPerson'], axis=1)

merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   RecordID                  704 non-null    int64  
 1   ZipCode                   704 non-null    int64  
 2   City                      704 non-null    object 
 3   County                    704 non-null    object 
 4   TotalPopulation           704 non-null    int64  
 5   MedianAge                 704 non-null    float64
 6   AvgHouseholdSize          704 non-null    float64
 7   NetMigration              704 non-null    float64
 8   AvgFinalValue             704 non-null    float64
 9   PopulationDensity         704 non-null    float64
 10  BusinessResidentialRatio  704 non-null    float64
 11  MostCommonRace            704 non-null    object 
 12  WealthIndex               704 non-null    float64
 13  HigherEducationRatio      704 non-null    float64
dtypes: float64

In [123]:
merged_df['SuccessScore'] = 0

# 1. Total Population (higher is more successful) - top 33%
pop_threshold = merged_df['TotalPopulation'].quantile(0.66)
merged_df.loc[merged_df['TotalPopulation'] > pop_threshold, 'SuccessScore'] += 1

# 2. Median Age (age = 13 to 50 is the most successful)
merged_df.loc[(merged_df['MedianAge'] >= 13) & (merged_df['MedianAge'] <= 50), 'SuccessScore'] += 1

# 3. Avg Household Size (higher is more successful) - top 33%
household_size_threshold = merged_df['AvgHouseholdSize'].quantile(0.66)
merged_df.loc[merged_df['AvgHouseholdSize'] > household_size_threshold, 'SuccessScore'] += 1

# 4. Net Migration (higher is more successful) - top 33%
net_migration_threshold = merged_df['NetMigration'].quantile(0.66)
merged_df.loc[merged_df['NetMigration'] > net_migration_threshold, 'SuccessScore'] += 1

# 5. Avg Final Value ($1M+ house prices is most favorable)
merged_df.loc[merged_df['AvgFinalValue'] >= 1000000, 'SuccessScore'] += 1

# 6. Population Density (higher is more favorable) - top 33%
density_threshold = merged_df['PopulationDensity'].quantile(0.66)
merged_df.loc[merged_df['PopulationDensity'] > density_threshold, 'SuccessScore'] += 1

# 7. Business Residential Ratio (higher is favorable) - top 33%
ratio_threshold = merged_df['BusinessResidentialRatio'].quantile(0.66)
merged_df.loc[merged_df['BusinessResidentialRatio'] > ratio_threshold, 'SuccessScore'] += 1

# 8. Most Common Race (white ppl unless the majority is a minority) - simplified for now to just 'White'
merged_df.loc[merged_df['MostCommonRace'] == 'White', 'SuccessScore'] += 1

# 9. Wealth Index (higher is favorable) - top 33%
wealth_threshold = merged_df['WealthIndex'].quantile(0.66)
merged_df.loc[merged_df['WealthIndex'] > wealth_threshold, 'SuccessScore'] += 1

# 10. Higher Education Ratio (higher is more favorable) - top 66%
edu_ratio_threshold = merged_df['HigherEducationRatio'].quantile(0.33)
merged_df.loc[merged_df['HigherEducationRatio'] > edu_ratio_threshold, 'SuccessScore'] += 1

# Create the binary 'Success' target variable based on a threshold of the SuccessScore
# Let's say a location is 'successful' if it meets at least 5 out of 10 criteria
success_criteria_count = 10 # Total number of criteria applied
success_threshold_score = success_criteria_count * 0.5 # e.g., 50% of criteria met
merged_df['Success'] = (merged_df['SuccessScore'] >= success_threshold_score).astype(int)

print("Distribution of SuccessScore:")
print(merged_df['SuccessScore'].value_counts())
print("\nDistribution of binary Success variable:")
print(merged_df['Success'].value_counts())
print("\nDataFrame head with new SuccessScore and Success columns:")
print(merged_df[['ZipCode', 'SuccessScore', 'Success']].head())

Distribution of SuccessScore:
SuccessScore
5    232
4    216
6    116
3    109
7     23
2      6
8      2
Name: count, dtype: int64

Distribution of binary Success variable:
Success
1    373
0    331
Name: count, dtype: int64

DataFrame head with new SuccessScore and Success columns:
   ZipCode  SuccessScore  Success
0    90001             5        1
1    90001             5        1
2    90002             4        0
3    90002             4        0
4    90003             4        0


In [124]:
merged_df.head(500)

,RecordID,ZipCode,City,County,State_zip,Latitude,Longitude,DominantAreaCode,ResidentialDeliveries,ResidentialPOBoxes,...,Latitude_property,Longitude_property,PopulationDensity,AvgIncomePerPerson,BusinessResidentialRatio,MostCommonRace,WealthIndex,HigherEducationRatio,SuccessScore,Success
0,1,90001,FIRESTONE PARK,LOS ANGELES,CA,33.9740,-118.2494,323,13443,304,...,33.720088,-117.838963,3.906439,16530.0,0.109045,Hispanic,669806.0,0.065738,5,1
1,2,90001,LOS ANGELES,LOS ANGELES,CA,33.9740,-118.2494,323,13443,304,...,33.720088,-117.838963,3.906439,16530.0,0.109045,Hispanic,669806.0,0.065738,5,1
2,3,90002,LOS ANGELES,LOS ANGELES,CA,33.9493,-118.2462,323,12443,177,...,33.720088,-117.838963,4.217122,15998.0,0.036323,Hispanic,652159.0,0.059906,4,0
3,4,90002,WATTS,LOS ANGELES,CA,33.9493,-118.2462,323,12443,177,...,33.720088,-117.838963,4.217122,15998.0,0.036323,Hispanic,652159.0,0.059906,4,0
4,5,90003,BROADWAY MANCHESTER,LOS ANGELES,CA,33.9634,-118.2742,323,18149,69,...,33.720088,-117.838963,3.800546,15605.0,0.090028,Hispanic,672733.0,0.059901,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,91733,SOUTH EL MONTE,LOS ANGELES,CA,34.0540,-118.0456,626,10512,99,...,33.720088,-117.838963,3.308109,19941.0,0.264530,Hispanic,770375.0,0.095955,5,1
496,497,91740,GLENDORA,LOS ANGELES,CA,34.1166,-117.8532,626,8902,882,...,33.720088,-117.838963,2.746414,36165.0,0.119735,White,854975.0,0.244522,5,1
497,498,91741,GLENDORA,LOS ANGELES,CA,34.1391,-117.8564,626,9728,0,...,33.720088,-117.838963,2.567935,50749.0,0.059102,White,1042446.0,0.289029,4,0
498,499,91744,CITY OF INDUSTRY,LOS ANGELES,CA,34.0310,-117.9406,626,18581,134,...,33.720088,-117.838963,4.070418,22965.0,0.072974,Hispanic,756266.0,0.112350,4,0


# **Model Training**

In [128]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd # Import pandas if not already imported

# Store ZipCode before dropping it from features
zipcodes = merged_df['ZipCode']

# Define features (X) and target (y)
# Drop 'Success' as it's derived from SuccessScore and other non-predictive columns like RecordID
X = merged_df.drop(columns=['SuccessScore', 'Success', 'RecordID', 'ZipCode'])
y = merged_df['SuccessScore']

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=np.number).columns

# Create a column transformer for one-hot encoding categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep numerical columns as they are
)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Start of added code for error diagnosis and handling ---

# Define min/max values for float32
float32_max = np.finfo(np.float32).max
float32_min = np.finfo(np.float32).min

# Process numerical columns to handle inf, NaNs, and values outside float32 range
for col in numerical_features:
    # 1. Handle infinite values: Replace inf/-inf with NaN
    if np.isinf(X_train[col]).any():
        print(f"Column '{col}' contains infinite values. Replacing with NaN.")
        X_train[col] = X_train[col].replace([np.inf, -np.inf], np.nan)
        X_test[col] = X_test[col].replace([np.inf, -np.inf], np.nan)

    # 2. Fill NaN values (original NaNs or those converted from inf) with median
    if X_train[col].isnull().any():
        median_val = X_train[col].median()
        print(f"Column '{col}' contains NaN values. Filling with median: {median_val}")
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

    # 3. Cap excessively large/small values to float32 limits
    if (X_train[col].max() > float32_max) or (X_train[col].min() < float32_min):
        print(f"Column '{col}' contains values outside float32 range (max: {X_train[col].max()}, min: {X_train[col].min()}). Capping to float32 limits.")
        X_train[col] = np.clip(X_train[col], float32_min, float32_max)
        X_test[col] = np.clip(X_test[col], float32_min, float32_max)

# end--------

# Create a pipeline with preprocessing and RandomForestRegressor
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', RandomForestRegressor(random_state=42))])

# Train the model
model_pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model_pipeline.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared: {r2:.2f}")

# Create a DataFrame to compare actual and predicted SuccessScores
predictions_df = pd.DataFrame({
    'ZipCode': zipcodes.loc[y_test.index],
    'Actual_SuccessScore': y_test,
    'Predicted_SuccessScore': y_pred,
    'Actual_Score_0_100': y_test * 10,
    'Predicted_Score_0_100': y_pred * 10
})
display(predictions_df.head(100))

Mean Squared Error (MSE): 0.14
R-squared: 0.86


,ZipCode,Actual_SuccessScore,Predicted_SuccessScore,Actual_Score_0_100,Predicted_Score_0_100
296,90805,4,3.98,40,39.8
81,90038,4,4.19,40,41.9
77,90037,4,3.93,40,39.3
208,90292,5,5.00,50,50.0
318,91010,4,3.72,40,37.2
...,...,...,...,...,...
569,92656,5,4.98,50,49.8
631,92804,4,4.00,40,40.0
90,90042,4,3.99,40,39.9
181,90250,3,3.08,30,30.8


In [129]:
from sklearn.metrics import mean_absolute_error

# Calculate MAE
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared: {r2:.2f}")

Mean Squared Error (MSE): 0.14
Mean Absolute Error (MAE): 0.22
R-squared: 0.86


In [130]:
# Calculate and display mean and median for success scores
print("\nDescriptive Statistics for Success Scores:")
print("\nOriginal 0-10 Scale:")
print(predictions_df[['Actual_SuccessScore', 'Predicted_SuccessScore']].agg(['mean', 'median', 'min', 'max']))

print("\nScaled 0-100 Scale:")
print(predictions_df[['Actual_Score_0_100', 'Predicted_Score_0_100']].agg(['mean', 'median', 'min', 'max']))


Descriptive Statistics for Success Scores:

Original 0-10 Scale:
        Actual_SuccessScore  Predicted_SuccessScore
mean                4.48227                4.462695
median              4.00000                4.380000
min                 2.00000                2.770000
max                 7.00000                6.950000

Scaled 0-100 Scale:
        Actual_Score_0_100  Predicted_Score_0_100
mean             44.822695               44.62695
median           40.000000               43.80000
min              20.000000               27.70000
max              70.000000               69.50000
